# 07 — Two APIs, one assistant

**Goal:** the multi-API moment: route an intent to the right surface first, then to the right operation — and refuse intents neither surface supports.

**Fixtures:** petstore-mini + weather-mini.

In [ ]:
# Offline by default. Repo + fixture paths:
from pathlib import Path
import sys

here = Path.cwd()
while not (here / "pyproject.toml").exists():
    here = here.parent
sys.path.insert(0, str(here / "src"))
FIXTURES = here / "cookbook" / "fixtures"
CORPUS_DIR = here / "data" / "corpus"
print(f"fixtures: {FIXTURES}")

## 1. Load both comprehended surfaces

In [ ]:
import yaml

def load_surface(name: str) -> dict:
    spec = yaml.safe_load((FIXTURES / name).read_text())
    return {
        'title': spec['info']['title'],
        'ops': {
            op['operationId']: f"{op.get('summary','')} ({method.upper()} {path})"
            for path, methods in spec['paths'].items()
            for method, op in methods.items()
        },
    }

surfaces = {
    'petstore': load_surface('petstore-mini.yaml'),
    'weather': load_surface('weather-mini.yaml'),
}
for key, surface in surfaces.items():
    print(f"{key}: {surface['title']} — {len(surface['ops'])} operations")

## 2. Two-level routing: surface, then operation

With one API you route to an operation. With N APIs the FIRST decision is which surface — and the refusal case ('neither') becomes more important, not less: more tools means more plausible-looking wrong choices.

In [ ]:
def route(intent: str) -> str:
    lowered = intent.lower()
    best = (0, None, None)
    for surface_key, surface in surfaces.items():
        for op_id, description in surface['ops'].items():
            text = f"{op_id} {description}".lower()
            overlap = sum(1 for token in lowered.split() if token in text)
            if overlap > best[0]:
                best = (overlap, surface_key, op_id)
    score, surface_key, op_id = best
    if score < 2:   # a routing FLOOR: one stray word is not a match
        return f'REFUSE: no surface supports {intent!r} (best score {score})'
    return f'{surface_key}.{op_id} — {surfaces[surface_key]["ops"][op_id]}'

for intent in (
    'what is the weather forecast for Lisbon this week',
    'order two of pet 7',
    'transfer money to my cousin',
):
    print(f'{intent!r}\n  -> {route(intent)}\n')

## 3. The routing floor is a safety control

Note the third intent: with no floor, 'transfer' might weakly match something and the assistant would confidently do the wrong thing. A **minimum-evidence threshold before acting** is the multi-API version of the refusal path you built in Week 1.

## 4. Live version

Add both fixtures with `gecko add`, serve, and connect (notebooks 02/04): your assistant now sees both surfaces' question-shaped tools and does this routing natively — with Gecko's comprehension quality instead of our toy keyword overlap.